In [1]:
import sys
import os
# sys.path.append('..')
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, PROJECT_ROOT)

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.tensorboard import SummaryWriter
import numpy as np
import pandas as pd

import datetime
from src.models.autoencoder import Autoencoder
from src.data.load_cifar10 import get_cifar10_loaders, create_and_load_subset_c10
from src.data.load_cifar100 import get_cifar100_loaders, create_and_load_subset
from src.data.CovidDataset import CovidDataset
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cuda


In [2]:
# Model
model = Autoencoder(latent_dim=256).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [3]:
# Dane
# train_loader, val_loader, test_loader = get_cifar10_loaders(batch_size=64)
# train_loader, val_loader, test_loader = get_cifar100_loaders(batch_size=64)
# _, selected_classes, train_loader, val_loader, test_loader = create_and_load_subset(
#     num_classes=2, 
#     batch_size=64
# )

_, selected_classes, train_loader, val_loader, test_loader = create_and_load_subset_c10(
    num_classes=2, 
    batch_size=64
)
print(f"Trenowanie na klasach: {selected_classes}")


Wylosowano nowe klasy: [3, 0]
Trenowanie na klasach: [3, 0]


https://www.datacamp.com/tutorial/pytorch-cnn-tutorial

In [ ]:
# Trening

BASE_DIR = os.getcwd()
save_dir = os.path.join(BASE_DIR, '..', 'training_results', 'autoencoder', 'cifar10',  f'{len(selected_classes)}_classes')
# save_dir = os.path.join(BASE_DIR, '..', 'training_results', 'autoencoder', 'cifar100', f'{len(selected_classes)}_classes')
print(save_dir)
writer_dir = os.path.join(BASE_DIR, '..', 'training_results', 'tb', 'cifar10')
# writer_dir = os.path.join(BASE_DIR, '..', 'training_results', 'tb', 'cifar100' , f'{len(selected_classes)}_classes')
os.makedirs(save_dir, exist_ok=True)
os.makedirs(writer_dir, exist_ok=True)
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
writer = SummaryWriter(os.path.join(writer_dir, f'autoencoder_{timestamp}'))

num_epochs = 250

train_losses = []
val_losses = []
epoch_number = 0
best_val_loss = float('inf')
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    
    for batch_idx, (image, _) in enumerate(train_loader):
        image = image.to(device)
        # target = target.to(device)

        outputs = model(image)
        loss = criterion(outputs, image)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        if batch_idx % 100 == 0:
            print(f"  [{epoch+1}/{num_epochs}] Batch {batch_idx}/{len(train_loader)} "
                  f"Loss: {loss.item():.4f}")
            
    # Średni train loss
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    writer.add_scalar('Loss/train', train_loss, epoch)

    # validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for image, _ in val_loader:
            image = image.to(device)
            # target = target.to(device)

            outputs = model(image)
            loss = criterion(outputs, image)
            val_loss += loss.item()
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    writer.add_scalar('Loss/val', val_loss, epoch)
    
    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")


    # save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'latent_dim': 256,
            'train_losses': train_losses,
            'val_losses': val_losses,
            'selected_classes': selected_classes,
        }
        timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        checkpoint_path = os.path.join(save_dir, f'autoencoder_cifar10_best_{timestamp}.pt')
        # checkpoint_path = os.path.join(save_dir, f'autoencoder_cifar100_best_{timestamp}.pt')
        torch.save(checkpoint, checkpoint_path)


df = pd.DataFrame({
    'epoch': range(1, num_epochs + 1),
    'train_loss': train_losses,
    'val_loss': val_losses
})
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
history_csv = os.path.join(save_dir, f'autoencoder_cifar10_training_results_{timestamp}.csv')
# history_csv = os.path.join(save_dir, f'autoencoder_cifar100_training_results_{timestamp}.csv')
df.to_csv(history_csv, index=False)


C:\Users\martu\Desktop\studia\magisterka\2sem\ml_projekt\src\notebooks_test_train\..\training_results\autoencoder\cifar10\2_classes
  [1/250] Batch 0/125 Loss: 0.0644
  [1/250] Batch 100/125 Loss: 0.0329
Epoch [1/250]
Train Loss: 0.0379
Val Loss:   0.0333
  [2/250] Batch 0/125 Loss: 0.0318
  [2/250] Batch 100/125 Loss: 0.0370


 # Final evaluation 

In [38]:
save_dir = os.path.join(os.getcwd(), '..', 'training_results', 'autoencoder', 'cifar100', '5_classes')
best_checkpoint = torch.load(os.path.join(save_dir, 'autoencoder_cifar100_best_20260113_185102.pt'))
saved_classes = best_checkpoint['selected_classes']
_, _, _, _, test_loader = create_and_load_subset(
    selected_classes=saved_classes,
    num_classes=5, 
    batch_size=64
)

# save_dir = os.path.join(os.getcwd(), '..', 'training_results', 'autoencoder', 'cifar100')
# best_checkpoint = torch.load(os.path.join(save_dir, 'autoencoder_cifar100_best_20260107_170204.pt'))
model.load_state_dict(best_checkpoint['model_state_dict'])
correct_test = 0
total_test = 0
test_loss = 0

model.eval()
print("\nCalculating metrics on test set...")

with torch.no_grad():
    for images, _ in test_loader:
        images= images.to(device)
        outputs = model(images)
        loss = criterion(outputs, images)
        test_loss += loss.item()
        
test_loss /= len(test_loader)    
# writer.add_scalar('Loss/test', test_loss)
print("FINAL EVALUATION RESULTS")
# print(f"Best Val Loss:       {best_val_loss:.6f}")
print(f"Test Loss:           {test_loss:.6f}")      
        
    

Używam podanych klas: [80, 5, 97, 13, 52]

Calculating metrics on test set...
FINAL EVALUATION RESULTS
Test Loss:           0.021415
